<div dir="rtl">

# Pony ORM: ימי שיתוף אופניים ומדינות של הבנק העולמי

</div>

<div dir="rtl">

**שני שיעורים, מחברת אחת.** המחברת הזו עוקבת אחרי שני שיעורים של הקורס, זה אחר זה: חלק 1, [טבלאות כאובייקטים ב־Python: היכרות ראשונה עם Pony ORM](https://dvirross.com/he/courses/intro-data-science/python-orm-pony) (משימות 1 עד 7); חלק 2, [קשרי Pony ORM: טבלה שמצביעה על טבלה אחרת](https://dvirross.com/he/courses/intro-data-science/python-orm-pony-relations) (משימות 8 עד 11). כותרת מסמנת את המקום שבו כל חלק מתחיל. המשימות ממוספרות מ־1 עד 11 בלי הפסקה, ולכן `check(N)` שייכת תמיד למשימה N של החלק שמחזיק אותה.

</div>

<div dir="rtl">

# חלק 1: טבלאות כאובייקטים ב־Python: היכרות ראשונה עם Pony ORM

דף השיעור של החלק הזה: [טבלאות כאובייקטים ב־Python: היכרות ראשונה עם Pony ORM](https://dvirross.com/he/courses/intro-data-science/python-orm-pony)

</div>

<div dir="rtl">

## האם תוכנית יכולה להשתמש בשורות של טבלה כבאובייקטים רגילים?

ב[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python) התוכנית שלחה SQL כטקסט וקיבלה בחזרה טאפלים. **[ORM](https://dvirross.com/he/courses/intro-data-science/glossary#orm)** (מיפוי בין טבלאות לאובייקטים) משנה את חלוקת העבודה: מתארים טבלה כ[**מחלקה**](https://dvirross.com/he/courses/intro-data-science/glossary#class) ב־Python, כל שורה הופכת ל[**אובייקט**](https://dvirross.com/he/courses/intro-data-science/glossary#object) של המחלקה הזו, וה־ORM כותב בשבילכם את ה־SQL. השיעור הזה משתמש ב־Pony ORM, ספרייה של Python. הוא עובד על ימי שיתוף האופניים, כמו קודם. טבלאות שמצביעות זו על זו, עם נתונים מהבנק העולמי, הן הנושא של [קשרי Pony ORM: טבלה שמצביעה על טבלה אחרת](https://dvirross.com/he/courses/intro-data-science/python-orm-pony-relations), שממשיך מהשיעור הזה.

ל־Pony יש תכסיס מיוחד שמתחבר ל[אובייקטים וביטויים יוצרים](https://dvirross.com/he/courses/intro-data-science/python-objects-and-generators): כותבים שאילתה כ[ביטוי יוצר](https://dvirross.com/he/courses/intro-data-science/glossary#generator-expression) (גנרטור), כמו:

```python
d for d in Day if d.season == 'Summer'
```

Pony קורא את הביטוי ומתרגם אותו ל־SQL. לכן השיעור ההוא השקיע זמן בצורה של הביטוי.

</div>

<div dir="rtl">

**איך עובדים:** הריצו את התאים מלמעלה למטה, כי התאים חולקים את המשתנים שלהם. תא שכתוב בו `# Your turn` ריק או מחזיק את ההכנה של המשימה: כתבו בו את הקוד, הריצו אותו, והריצו את התא שמתחתיו, שבודק את התשובה בלי להראות את הפתרון. במחברת עם הפתרונות יש מתחת לכל משימה את הפתרון, את הטעויות הנפוצות ואת הפלט הצפוי. אובייקט מסד נתונים והישויות שלו מוגדרים פעם אחת בכל הרצה: אם מריצים בפעם השנייה את התא שמגדיר את `db` או את `Day`, Pony נעצר בשגיאה, ולכן מפעילים מחדש את סביבת ההרצה (ב־Colab: **Runtime**, ואז **Restart session**) ומריצים שוב את התאים מההתחלה.

</div>

<div dir="rtl">

## הכנה: להתקין את Pony ולהביא את קובץ הנתונים

אין צורך בחשבון ואין צורך במפתח. התא הבא מתקין את Pony, הספרייה של השיעור הזה, אם הסביבה עוד לא כוללת אותה (Colab אינו כולל אותה, וההתקנה לוקחת כמה שניות). התא שאחריו מוריד מהאתר הציבורי את קובץ ה־CSV הקטן של השיעור ושומר אותו ליד המחברת (ב־Colab הוא מופיע בחלונית **Files**). אם ההורדה נכשלת, פותחים את הכתובת שלמטה בדפדפן, מורידים את הקובץ ומעלים אותו לחלונית **Files** של Colab.

`https://dvirross.com/materials/courses/intro-data-science/world-bank-more-countries.csv`

</div>

In [1]:
try:
    import pony
except ImportError:
    !pip install -q pony
    import pony
print('Pony מוכן, גרסה', pony.__version__)

Pony מוכן, גרסה 0.7.20


In [2]:
import os
import urllib.request

FILE_URL = 'https://dvirross.com/materials/courses/intro-data-science/world-bank-more-countries.csv'
FILE_PATH = 'world-bank-more-countries.csv'

if not os.path.exists(FILE_PATH):
    # the site refuses the default user agent of Python, so the request names one
    request = urllib.request.Request(FILE_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, open(FILE_PATH, 'wb') as file:
        file.write(response.read())
print('קובץ הנתונים מוכן:', FILE_PATH)

קובץ הנתונים מוכן: world-bank-more-countries.csv


<div dir="rtl">

התא הבא מגדיר את הפונקציה `check(task_number)`. אחרי שמריצים את התא של משימה, מריצים את התא שמתחתיו: `check` מריצה שוב את התא שלכם, אוספת את מה שהוא מדפיס ומשווה אותו לפלט הצפוי. היא אומרת אם הפלט נכון, ואם לא באיזו שורה הוא שונה; היא לא מראה את הפתרון.

</div>

In [3]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        '2dd29fb6f518',
        'e1ae88365d47',
        '396a1bf61e9a'
    ],
    '2': [
        '6f42ae6ab479',
        '5c558c73b770'
    ],
    '3': [
        '2dd29fb6f518',
        'e1ae88365d47',
        '396a1bf61e9a'
    ],
    '4': [
        'f5ca38f748a1',
        'e7f6c011776e'
    ],
    '5': [
        '1d564088a318',
        'e18cad10fc80'
    ],
    '6': [
        '2b2bc2ccad4c',
        '51b2762f9f61'
    ],
    '7': [
        '59e1e5985094'
    ],
    '8': [
        'dc289ab8c8fa',
        '7b2c7298ef49',
        '401cb6939b81',
        '7413e6942dbb',
        '6cceb9bc987f',
        '5e0fdfdc44f9'
    ],
    '9': [
        '8ee58a0b6923',
        '770859d608eb'
    ],
    '10': [
        'ee7e860a857f'
    ],
    '11': [
        '809c025ba41f',
        'dc937b598926'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('אין משימה כזו: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('התא נכשל בשגיאה:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('לא הודפס כלום. הדפיסו את התשובה בתא המשימה, הריצו אותו, והריצו את `check` מיד אחריו.')
    elif len(got) != len(want):
        print('התא הדפיס {got} שורות והמשימה מצפה ל־{want}. קראו שוב את המשימה ואת הרמז.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('שורה {k} שונה מהפלט הצפוי. קראו שוב את המשימה ואת הרמז.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: הפלט כצפוי.')


<div dir="rtl">

## מתארים טבלה כמחלקה

מחלקה שמייצגת טבלה נקראת **[ישות](https://dvirross.com/he/courses/intro-data-science/glossary#entity)** (entity). היא יורשת מ־`db.Entity`, כש־`db` הוא האובייקט שמייצג את מסד הנתונים, וכל עמודה היא שורה ששמה את התכונה ואת סוגה: `Required(str)` הוא טקסט שחייב להיות לו ערך, ו־`Required(int)` הוא מספר שלם שחייב להיות לו ערך. השורה הראשונה בתיבה הבאה, `from pony.orm import *`, מביאה כל שם ש־Pony מציע. שלושה שמות של Pony עושים את ההכנה. `Database()` יוצרת את האובייקט `db` שמייצג מסד נתונים. `db.bind(...)` מחברת אותו ל־SQLite; `':memory:'` הוא שוב מסד זמני בזיכרון, כמו ב[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python). `db.generate_mapping(...)` משלימה את תיאור הטבלאות; עם `create_tables=True` היא גם יוצרת את הטבלה במסד, אם היא עוד לא קיימת.

</div>

In [4]:
from pony.orm import *

db = Database()
db.bind(provider='sqlite', filename=':memory:')

class Day(db.Entity):
    date = Required(str)
    season = Required(str)
    working_day = Required(int)
    rentals = Required(int)

db.generate_mapping(create_tables=True)
print(list(db.entities))

['Day']


<div dir="rtl">

קוראים את השורות לפי הסדר:

- `from pony.orm import *` מביאה את השמות של Pony. הכוכבית פירושה "כל שם שהמודול מציע", כמו ב[ערכים ואוספים](https://dvirross.com/he/courses/intro-data-science/python-values-and-collections). בין השמות גם `sum`, `min` ו־`max`, שממשיכות לעבוד על רשימות ומספרים רגילים כמו קודם.
- המחלקה `Day` מתארת את הטבלה `Day`: שורה לכל יום, עם העמודות `date`, `season`, `working_day` ו־`rentals`. Pony מוסיף גם עמודה `id` שממספרת את השורות. זה ה[**מפתח הראשי**](https://dvirross.com/he/courses/intro-data-science/glossary#primary-key), ולכן לא היה צורך לכתוב אותו.
- `print(list(db.entities))` מדפיסה את השמות של הישויות שמסד הנתונים מכיר.

## מוסיפים שורות כאובייקטים

בתוכנית, Pony מדבר עם מסד הנתונים רק בתוך בלוק `with db_session:` (ראו [db_session](https://dvirross.com/he/courses/intro-data-science/glossary#db-session)). כשהבלוק מסתיים כרגיל, Pony שומר את השינויים; אם שגיאה עוצרת את הבלוק, הוא מבטל אותם. יצירת שורה היא יצירת אובייקט: קוראים למחלקה עם ערכי העמודות.

</div>

In [5]:
rows = [
    ('2011-01-01', 'Winter', 0, 985),
    ('2011-01-02', 'Winter', 0, 801),
    ('2011-01-03', 'Winter', 1, 1349),
    ('2011-07-04', 'Summer', 0, 6043),
    ('2011-07-05', 'Summer', 1, 4665),
    ('2011-07-06', 'Summer', 1, 4629),
]

with db_session:
    for date, season, working_day, rentals in rows:
        Day(
            date=date, season=season, working_day=working_day,
            rentals=rentals
        )
    print(Day.select().count())
    day = Day[1]
    print(day, day.date, day.rentals)

6
Day[1] 2011-01-01 985


<div dir="rtl">

הלולאה משתמשת ב[חילוץ](https://dvirross.com/he/courses/intro-data-science/glossary#unpacking) (unpacking) מ[לולאות ופונקציות](https://dvirross.com/he/courses/intro-data-science/python-loops-and-functions), ושש השורות הן הימים האמיתיים משיעורי ה־SQL. `Day[1]` שולפת את האובייקט שהמפתח הראשי שלו הוא 1, וקוראים את התכונות שלו בנקודה, כמו ב[אובייקטים וביטויים יוצרים](https://dvirross.com/he/courses/intro-data-science/python-objects-and-generators). הדפסת האובייקט מראה את שם המחלקה ואת המפתח. השתמשו באובייקטים בתוך הבלוק `with db_session:`, שבו Pony מחובר למסד הנתונים.

### מה קורה מחוץ לבלוק db_session

</div>

<div dir="rtl">

כל שימוש בישויות שייך לתוך `with db_session:` בתוכנית ש־Python מריצה מקובץ: שאילתה מחוץ לבלוק נעצרת בשגיאה `TransactionError: db_session is required when working with the database`. מחברת מתנהגת אחרת, כפי שההסבר שלמטה מפרט, ולכן התאים במחברת הזו משאירים את הבלוק בכל שאילתה, כמו בדף.

</div>

<div dir="rtl">

**Pony במחברת (Jupyter או Colab)**

</div>

<div dir="rtl">

Pony מחליט עד כמה להקפיד על `db_session` ברגע שמייבאים אותו בפעם הראשונה. תוכנית ש־Python מריצה מקובץ אינה אינטראקטיבית, ו־Pony דורש בה את הבלוק. מחברת (Jupyter או Colab) היא קונסולה **אינטראקטיבית**: Pony מזהה זאת ומאפשר להריץ שאילתה גם מחוץ ל־`with db_session:`. אותה שורה `print(Day.select().count())` מדפיסה שם את מספר השורות במקום להיעצר בשגיאה.

נמשיך לכתוב `with db_session:` גם במחברת. הקוד ממשיך לעבוד בלי שינוי כשעוברים לתוכנית, וסוף הבלוק הוא המקום ש־Pony שומר בו את השינויים, או מבטל אותם אחרי שגיאה. ב־Colab מתקינים את Pony בתא עם `pip install pony`, כי Colab אינו כולל אותו.

</div>

<div dir="rtl">

## קוראים שורות: שאילתה כביטוי יוצר

`select(...)` מקבלת ביטוי יוצר. החלק שלפני `for` אומר מה להחזיר, החלק שאחרי `in` אומר מאיזו ישות, והתנאי אחרי `if` מסנן את השורות, בדיוק כמו `SELECT`, `FROM` ו־`WHERE`. מעבר בלולאה על התוצאה מריץ את השאילתה ונותן אובייקט אחד לכל שורה. השיטה `order_by` ממיינת את שורות התוצאה לפי הערך של הפונקציה הקטנה שנותנים לה, ו־`desc(...)` הופכת את הסדר לסדר מהגדול אל הקטן, כמו `ORDER BY ... DESC`.

</div>

In [6]:
with db_session:
    summer = select(d for d in Day if d.season == 'Summer')
    for d in summer.order_by(lambda d: desc(d.rentals)):
        print(d.date, d.rentals)

2011-07-04 6043
2011-07-05 4665
2011-07-06 4629


<div dir="rtl">

כדי לראות את ה־SQL ש־Pony כתב, השיטה `get_sql` של שאילתה מחזירה אותו כטקסט:

</div>

In [7]:
with db_session:
    print(
        select(
            d for d in Day if d.season == 'Summer'
        ).get_sql()
    )

SELECT "d"."id", "d"."date", "d"."season", "d"."working_day", "d"."rentals"
FROM "Day" "d"
WHERE "d"."season" = 'Summer'


<div dir="rtl">

זו שאילתה שהייתם יכולים להקליד ב־DB Browser. Pony משתמש גם ב[**פרמטרים**](https://dvirross.com/he/courses/intro-data-science/glossary#argument) מ[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python): משתנה של Python בתוך הביטוי אינו מודבק לטקסט, אלא נשלח כערך עבור סימן שאלה.

</div>

In [8]:
with db_session:
    limit = 4600
    busy = select(d for d in Day if d.rentals > limit)
    print(busy.get_sql())
    print([d.date for d in busy])

SELECT "d"."id", "d"."date", "d"."season", "d"."working_day", "d"."rentals"
FROM "Day" "d"
WHERE "d"."rentals" > ?
['2011-07-04', '2011-07-05', '2011-07-06']


<div dir="rtl">

**[חיתוך](https://dvirross.com/he/courses/intro-data-science/glossary#slice)** (slice) לוקח רק את השורות הראשונות, כמו ברשימה: `select(...)[:2]` מחזירה את שני האובייקטים הראשונים.

כשצריך רק אובייקט אחד, השיטה `first()` קצרה מ־`list(...)[0]`. היא מחזירה את האובייקט הראשון בתוצאה, ו־`None` כשהתוצאה ריקה, ואילו `list(...)[0]` עוצרת בשגיאה על רשימה ריקה. היא שונה גם מהחיתוך `[:1]`: החיתוך מחזיר רשימה שיש בה לכל היותר אובייקט אחד, ו־`first()` מחזירה את האובייקט עצמו. אחרי `order_by(...)` האובייקט הראשון הוא הראשון לפי אותו סדר. התיבה לוקחת את היום עם הכי מעט השכרות, ואחר כך מבקשת יום עם יותר מ־10000 השכרות, שאינו קיים:

</div>

In [9]:
with db_session:
    quietest = select(d for d in Day).order_by(
        lambda d: d.rentals
    ).first()
    print(quietest.date, quietest.rentals)
    huge = select(d for d in Day if d.rentals > 10000).first()
    print(huge)

2011-01-02 801
None


<div dir="rtl">

הקריאה הראשונה מחזירה אובייקט, שאת התכונות שלו קוראים בנקודה, והשנייה אינה מוצאת יום ומחזירה `None`, ולכן בודקים אם התוצאה היא `None` לפני שקוראים ממנה תכונה.

### תרגול

**נסו בעצמכם.** בתוך בלוק `with db_session:` הדפיסו את התאריכים של ימי `'Winter'`, כל אחד בשורה משלו, מהתאריך המוקדם ביותר ועד המאוחר ביותר.

</div>

<div dir="rtl">

> **רמז**
>
> השאירו את ימי החורף עם תנאי אחרי `if`, מיינו את התוצאה עם `order_by(lambda d: d.date)`, ועברו עליה בלולאה כדי להדפיס תכונה אחת בכל שורה.

</div>

<div dir="rtl">

**פתרון**

</div>

In [10]:
with db_session:
    winter = select(d for d in Day if d.season == 'Winter')
    for d in winter.order_by(lambda d: d.date):
        print(d.date)

2011-01-01
2011-01-02
2011-01-03


<div dir="rtl">

**פלט צפוי**

```text
2011-01-01
2011-01-02
2011-01-03
```

</div>

In [11]:
check(1)

OK: הפלט כצפוי.


<div dir="rtl">

בחורף שלושה ימים, ולכן מודפסים שלושה תאריכים לפי סדר התאריכים.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    winter = select(d for d in Day if d.season == 'Winter')
    for d in winter.order_by(lambda d: d.rentals):
        print(d.date)
```

`order_by` ממיין לפי הביטוי שנותנים לו. כאן הוא ממיין לפי מספר ההשכרות, ולכן שלושת התאריכים מודפסים מהיום עם הכי מעט השכרות ליום עם הכי הרבה, ולא לפי סדר התאריכים.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    winter = select(d for d in Day if d.season == 'Winter')
    print([d.date for d in winter.order_by(lambda d: d.date)])
```

חסרה לולאה. `print` נקראת פעם אחת, עם הרשימה כולה, ולכן התאריכים מופיעים בשורה אחת בתוך סוגריים מרובעים במקום אחד בכל שורה.

</div>

<div dir="rtl">

## סיכומים וקבוצות

לפונקציות הסיכום של SQL יש שמות גם ב־Pony: `count`, `sum`, `avg`, `min` ו־`max`. כל אחת מקבלת ביטוי יוצר שנותן את הערכים לסיכום.

</div>

In [12]:
with db_session:
    print(count(d for d in Day))
    print(avg(d.rentals for d in Day if d.season == 'Summer'))
    print(max(d.rentals for d in Day))
    print(sum(d.rentals for d in Day if d.working_day == 1))

6
5112.333333333333
6043
10643


<div dir="rtl">

כשהביטוי מחזיר זוג, למשל העונה וסיכום, Pony מקבץ לפי החלק שאינו מסוכם:

</div>

In [13]:
with db_session:
    by_season = select(
        (d.season, avg(d.rentals)) for d in Day
    )
    print(by_season.get_sql())
    print(list(by_season))

SELECT "d"."season", AVG("d"."rentals")
FROM "Day" "d"
GROUP BY "d"."season"
[('Summer', 5112.333333333333), ('Winter', 1045.0)]


<div dir="rtl">

אותה צורה סופרת את השורות של כל קבוצה: כש־`count(d)` נמצאת בזוג, ו־`d` הוא היום כולו, Pony סופר את הימים של כל עונה. הוא סופר כל יום פעם אחת, לפי המפתח הראשי, ולכן יש ב־SQL `COUNT(DISTINCT ...)`.

</div>

In [14]:
with db_session:
    days_per_season = select(
        (d.season, count(d)) for d in Day
    )
    print(days_per_season.get_sql())
    print(list(days_per_season))

SELECT "d"."season", COUNT(DISTINCT "d"."id")
FROM "Day" "d"
GROUP BY "d"."season"
[('Summer', 3), ('Winter', 3)]


<div dir="rtl">

ב־SQL יש `GROUP BY` שלא כתבתם: הוא נלמד ב[מדריך ה־SQL, חלק א׳](https://dvirross.com/he/courses/intro-data-science/sql-in-db-browser), וכאן ה־ORM מוסיף אותו מתוך צורת הביטוי.

### אותו סכום בשתי דרכים

שאלה על סכום אפשר לענות עליה בלולאה ב־Python או בפונקציית סיכום, ושתי הדרכים נותנות אותו מספר. השיטה `select()` של ישות, בלי ארגומנטים, מחזירה את כל האובייקטים שלה. הלולאה שלהלן לוקחת כל יום ממסד הנתונים ומחברת את ההשכרות של ימי הקיץ בזה אחר זה, ב־Python. פונקציית הסיכום מבקשת את הסכום ממסד הנתונים:

</div>

In [15]:
with db_session:
    total = 0
    for d in Day.select():
        if d.season == 'Summer':
            total = total + d.rentals
    print(total)
    print(sum(d.rentals for d in Day if d.season == 'Summer'))

15337
15337


<div dir="rtl">

הלולאה מביאה את כל הימים אל Python, ו־`sum(...)` גורמת ל־Pony לשלוח `SELECT SUM(...)` אחד ולקבל מספר אחד. לשישה ימים שתי הדרכים לוקחות אותו זמן. בטבלה עם אלפי שורות הלולאה עדיין תעביר כל שורה אל Python, ו־`sum(...)` עדיין תקבל מספר אחד.

### סיכום שמזין שאילתה שנייה

סיכום הוא ערך רגיל של Python, ולכן אפשר לשמור אותו במשתנה ולהשתמש בו בשאילתה הבאה. השאילתה הראשונה שלהלן מוצאת את מספר ההשכרות הגדול ביותר בעזרת `max`, ואז `Day.get(rentals=top)` מחזירה את היום עם הערך הזה. השיטה `get` מצפה להתאמה אחת בדיוק: כששני ימים חולקים את הערך, Pony עוצר בשגיאה, ו־`select(...).first()` היא הדרך לקחת אחד מהם. השאילתה השנייה מוצאת בעזרת `avg` את ממוצע ההשכרות של כל הימים, ושאילתה משאירה את הימים שמעליו. Pony שולח את הערך השמור כפרמטר, כמו בשאילתה עם `limit` שלמעלה.

</div>

In [16]:
with db_session:
    top = max(d.rentals for d in Day)
    busiest = Day.get(rentals=top)
    print(busiest.date, top)

    mean = avg(d.rentals for d in Day)
    above = select(d for d in Day if d.rentals > mean)
    print(round(mean), above.count())

2011-07-04 6043
3079 3


<div dir="rtl">

הממוצע של ששת הימים הוא כ־3079, ולשלושה ימים יש יותר השכרות ממנו: שלושת ימי הקיץ.

### תרגול

**נסו בעצמכם.** בתוך בלוק `with db_session:` הדפיסו את מספר ההשכרות הגדול ביותר בין ימי החורף בעזרת `max`, ובשורה הבאה את ממוצע ההשכרות של ימי החורף בעזרת `avg`.

</div>

<div dir="rtl">

> **רמז**
>
> כל פונקציית סיכום מקבלת ביטוי יוצר, ובשתי השורות צריך את אותו תנאי אחרי `if`: העונה היא `'Winter'`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [17]:
with db_session:
    print(max(d.rentals for d in Day if d.season == 'Winter'))
    print(avg(d.rentals for d in Day if d.season == 'Winter'))

1349
1045.0


<div dir="rtl">

**פלט צפוי**

```text
1349
1045.0
```

</div>

In [18]:
check(2)

OK: הפלט כצפוי.


<div dir="rtl">

יום החורף העמוס ביותר הוא 1349, והממוצע של 985, 801 ו־1349 הוא 1045.0.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    print(max(d.rentals for d in Day))
    print(avg(d.rentals for d in Day if d.season == 'Winter'))
```

התנאי `if d.season == 'Winter'` חסר בשורה הראשונה, ולכן `max` מסתכלת על כל ששת הימים ומחזירה את המקסימום של הקיץ במקום של החורף. פונקציית סיכום מסתכלת רק על השורות שהביטוי מעביר.

</div>

<div dir="rtl">

**נסו בעצמכם.** בתוך בלוק `with db_session:` שמרו במשתנה את ממוצע ההשכרות של כל הימים. אחר כך הדפיסו את התאריכים של הימים עם פחות השכרות מהממוצע הזה, כל אחד בשורה משלו, מהתאריך המוקדם ביותר ועד המאוחר ביותר.

</div>

<div dir="rtl">

> **רמז**
>
> כתבו את הממוצע במשתנה בתוך הבלוק, השתמשו במשתנה אחרי `if` בתוך `select(...)`, ומיינו את התוצאה עם `order_by(lambda d: d.date)`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [19]:
with db_session:
    mean = avg(d.rentals for d in Day)
    below = select(d for d in Day if d.rentals < mean)
    for d in below.order_by(lambda d: d.date):
        print(d.date)

2011-01-01
2011-01-02
2011-01-03


<div dir="rtl">

**פלט צפוי**

```text
2011-01-01
2011-01-02
2011-01-03
```

</div>

In [20]:
check(3)

OK: הפלט כצפוי.


<div dir="rtl">

הממוצע של כל ששת הימים הוא כ־3079, ושלושת ימי החורף נמוכים ממנו.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    mean = avg(d.rentals for d in Day if d.season == 'Winter')
    below = select(d for d in Day if d.rentals < mean)
    for d in below.order_by(lambda d: d.date):
        print(d.date)
```

הממוצע נלקח רק מימי החורף, כ־1045, ולכן רק היום עם 801 השכרות נמוך ממנו. השאלה מבקשת את הממוצע של כל הימים.

</div>

<div dir="rtl">

## משנים ומוחקים שורות

כדי לשנות עמודה, מציבים ערך בתכונה של האובייקט. כדי למחוק שורה, קוראים ל־`delete()` של האובייקט שלה. קודם נוסיף **יום בדיקה** שאינו נתון אמיתי, כדי שהימים האמיתיים יישארו כפי שהם. `Day.get(...)` מחזירה את האובייקט היחיד שמתאים, או `None` אם אין כזה.

</div>

In [21]:
with db_session:
    Day(
        date='2099-01-01', season='Summer', working_day=1,
        rentals=1
    )

with db_session:
    test_day = Day.get(date='2099-01-01')
    test_day.rentals = 2
    print(test_day.rentals)

with db_session:
    print(Day.get(date='2099-01-01').rentals)
    Day.get(date='2099-01-01').delete()
    print(count(d for d in Day))
    print(Day.get(date='2099-01-01'))

2
2
6
None


<div dir="rtl">

הבלוק השני שינה את הערך והבלוק השלישי, שהוא שיחה חדשה, עדיין רואה אותו: השינוי נשמר כשהבלוק השני הסתיים. השוו ל[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python), שבו כתבתם `UPDATE` ו־`DELETE` כטקסט SQL והחלטתם מתי לקרוא ל־`commit()`. כאן סוף בלוק ה־`db_session` הוא ה־commit, ושגיאה בתוך הבלוק היא ה־rollback. אפשר גם לבטל בכוונה: הקריאה `rollback()` בתוך הבלוק מבטלת כל שינוי שנעשה מאז תחילת הבלוק, כמו ש־`conn.rollback()` עושה ב[סעיף על שינוי נתונים ושמירתם](https://dvirross.com/he/courses/intro-data-science/sql-from-python#שינוי-נתונים-ושמירתם), והבלוק ממשיך עם מסד הנתונים כפי שהיה.

הקריאה `commit()` בתוך הבלוק שומרת מיד את השינויים שנעשו עד כה, והבלוק ממשיך. `rollback()` שבאה אחריה מבטלת רק את מה שבא אחרי ה־`commit()`. התיבה שלהלן מוסיפה יום בדיקה, שומרת אותו עם `commit()`, מוסיפה יום בדיקה שני ומבטלת אותו עם `rollback()`. אחר כך בלוק חדש קורא מה מסד הנתונים מחזיק, ומוחק את יום הבדיקה הראשון, כדי שבטבלה יהיו שוב ששת הימים האמיתיים:

</div>

In [22]:
with db_session:
    Day(
        date='2097-01-01', season='Summer', working_day=1,
        rentals=1
    )
    commit()
    Day(
        date='2097-01-02', season='Summer', working_day=1,
        rentals=2
    )
    rollback()

with db_session:
    print(Day.get(date='2097-01-01').rentals)
    print(Day.get(date='2097-01-02'))
    Day.get(date='2097-01-01').delete()
    print(count(d for d in Day))

1
None
6


<div dir="rtl">

יום הבדיקה הראשון נשמר בידי `commit()`, ולכן הוא עדיין שם, והשני בוטל לפני שנשמר, ולכן `Day.get` לא מוצאת דבר ומחזירה `None`.

### תרגול

**נסו בעצמכם.** בבלוק `with db_session:` אחד צרו יום בדיקה עם התאריך `'2098-01-01'`, העונה `'Winter'`, `working_day` שווה 0 ו־10 השכרות. שנו את ההשכרות שלו ל־20 והדפיסו אותן. אחר כך מחקו את היום והדפיסו `count(d for d in Day)`.

</div>

<div dir="rtl">

> **רמז**
>
> שמרו במשתנה את האובייקט ש־`Day(...)` מחזירה. הציבו ערך בתכונה `rentals` כדי לשנות אותה, וקראו ל־`delete()` על האובייקט כדי להסיר את השורה.

</div>

<div dir="rtl">

**פתרון**

</div>

In [23]:
with db_session:
    day = Day(
        date='2098-01-01', season='Winter', working_day=0,
        rentals=10
    )
    day.rentals = 20
    print(day.rentals)
    day.delete()
    print(count(d for d in Day))

20
6


<div dir="rtl">

**פלט צפוי**

```text
20
6
```

</div>

In [24]:
check(4)

OK: הפלט כצפוי.


<div dir="rtl">

אחרי מחיקת היום יש בטבלה שוב ששת השורות המקוריות.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    day = Day(date='2098-01-01', season='Winter', working_day=0, rentals=10)
    day.rentals = 20
    print(day.rentals)
    print(count(d for d in Day))
    rollback()
```

היום לא נמחק, ולכן הספירה כוללת את השורה הנוספת. `rollback()` בסוף רק מבטלת את יום הבדיקה, כדי ששאר התיבות בדף ימשיכו לראות שישה ימים.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    day = Day(date='2098-01-01', season='Winter', working_day=0, rentals=10)
    rentals = day.rentals
    rentals = 20
    print(day.rentals)
    day.delete()
    print(count(d for d in Day))
```

השורה `rentals = 20` משנה משתנה נפרד שמחזיק עותק של המספר, ולא את התכונה של האובייקט. כדי לשנות את העמודה מציבים בתכונה: `day.rentals = 20`.

</div>

<div dir="rtl">

## תרגול

לכל שאלה יש תיבת קוד משלה: כתבו שם את התשובה ולחצו על **הריצו**. אם נתקעתם פתחו רמז, ואת הפתרון פתחו רק אחרי שניסיתם.

לפני השאלות, שימו לב מה נשאר זהה משיעורי ה־SQL: הטבלאות, המפתחות והחשיבה על אילו שורות ואיזה סיכום צריך. רק הדרך לכתוב את הבקשה משתנה.

**שאלה 1.** הדפיסו את התאריך ואת מספר ההשכרות של ימי העבודה (`working_day` הוא 1) עם יותר מ־4600 השכרות, מהיום העמוס ביותר ומטה.

</div>

<div dir="rtl">

> **רמז**
>
> שימו את שני התנאים אחרי `if`, מחוברים ב־`and`, וסדרו עם הפונקציה:
>
> ```python
> order_by(lambda d: desc(d.rentals))
> ```
>
> עברו בלולאה על התוצאה והדפיסו שתי תכונות.

</div>

<div dir="rtl">

**פתרון**

</div>

In [25]:
with db_session:
    days = select(
        d for d in Day
        if d.working_day == 1 and d.rentals > 4600
    )
    for d in days.order_by(lambda d: desc(d.rentals)):
        print(d.date, d.rentals)

2011-07-05 4665
2011-07-06 4629


<div dir="rtl">

**פלט צפוי**

```text
2011-07-05 4665
2011-07-06 4629
```

</div>

In [26]:
check(5)

OK: הפלט כצפוי.


<div dir="rtl">

אלה אותם שני ימים כמו ב[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python), וכעת הסינון נעשה במסד הנתונים.

</div>

<div dir="rtl">

**טעות נפוצה**

בלי `desc(...)`, `order_by` ממיין מהערך הקטן:

```python
with db_session:
    days = select(d for d in Day if d.working_day == 1 and d.rentals > 4600)
    for d in days.order_by(lambda d: d.rentals):
        print(d.date, d.rentals)
```

</div>

<div dir="rtl">

**שאלה 2.** הדפיסו לכל עונה כמה ימים היו בהם יותר מ־1000 השכרות, שורה לכל עונה: העונה, רווח, המספר.

</div>

<div dir="rtl">

> **רמז**
>
> החזירו זוג, את העונה ואת `count(d)`, ושימו את התנאי על `d.rentals` אחרי `if`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [27]:
with db_session:
    for season, number in select(
        (d.season, count(d)) for d in Day if d.rentals > 1000
    ):
        print(season, number)

Summer 3
Winter 1


<div dir="rtl">

**פלט צפוי**

```text
Summer 3
Winter 1
```

</div>

In [28]:
check(6)

OK: הפלט כצפוי.


<div dir="rtl">

רק יום חורף אחד עובר את 1000 ההשכרות, ושלושת ימי הקיץ כולם עוברים.

</div>

<div dir="rtl">

**טעות נפוצה**

בלי `if d.rentals > 1000` השאילתה סופרת את כל הימים של כל עונה:

```python
with db_session:
    for season, number in select((d.season, count(d)) for d in Day):
        print(season, number)
```

</div>

<div dir="rtl">

**שאלה 3.** הוסיפו את היום האמיתי `'2011-07-07'` (יום עבודה בקיץ עם 4592 השכרות), ואז הדפיסו את ממוצע ההשכרות של ימי הקיץ, מעוגל למספר שלם. סיימו עם `rollback()`, שמבטלת את היום החדש, כדי ששאר התיבות בדף ימשיכו לראות את אותם שישה ימים.

</div>

<div dir="rtl">

> **רמז**
>
> צרו את האובייקט עם `Day(...)` בתוך בלוק `db_session`, ואז קראו ל־`avg` על ביטוי יוצר שעוסק בימי הקיץ. עטפו אותו ב־`round(...)`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [29]:
with db_session:
    Day(
        date='2011-07-07', season='Summer', working_day=1,
        rentals=4592
    )
    print(
        round(
            avg(
                d.rentals for d in Day if d.season == 'Summer'
            )
        )
    )
    rollback()

4982


<div dir="rtl">

**פלט צפוי**

```text
4982
```

</div>

In [30]:
check(7)

OK: הפלט כצפוי.


<div dir="rtl">

Pony שולח את השורה החדשה למסד לפני שהוא מריץ את השאילתה, ולכן הממוצע כולל את היום שהרגע נוסף. אחר כך `rollback()` מבטלת את היום החדש.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    print(
        round(
            avg(d.rentals for d in Day if d.season == 'Summer')
        )
    )
    Day(
        date='2011-07-07', season='Summer', working_day=1,
        rentals=4592
    )
    rollback()
```

הממוצע מחושב לפני שהיום החדש נוסף, ולכן הוא מכסה רק את שלושת ימי הקיץ שכבר היו. צריך ליצור את היום קודם, ורק אז השאילתה רואה אותו.

</div>

<div dir="rtl">

## לזכור

- **ORM** ממפה טבלה למחלקה ושורה לאובייקט. **Pony** כותב את ה־SQL מתוך ביטוי יוצר, ולכן הצורה של הביטוי חשובה.
- מחלקות מסוג `db.Entity` מתארות טבלאות, `Required(...)` מתארת עמודה, ו־`db.generate_mapping(...)` יוצרת את הטבלאות.
- עובדים עם מסד הנתונים בתוך `with db_session:`: יצירת אובייקטים, הרצת שאילתות וקריאת אובייקטים קשורים. תוכנית שרצה מחוץ לבלוק נעצרת בשגיאה. סוף הבלוק שומר את השינויים, ושגיאה בתוכו מבטלת אותם.
- `select(...)`, `count`, `avg`, `sum`, `min` ו־`max` מתאימות ל־`SELECT`, `COUNT`, `AVG`, `SUM`, `MIN` ו־`MAX`; `get_sql()` מראה את ה־SQL ש־Pony כתב.
- כדי לשנות עמודה מציבים ערך בתכונה; כדי למחוק שורה קוראים ל־`delete()`.
- `commit()` בתוך הבלוק שומרת את השינויים עד כה; `rollback()` שבאה אחריה מבטלת רק את מה שבא אחרי ה־`commit()`.
- אפשר לשמור סיכום במשתנה ולהשתמש בו בשאילתה שנייה; `get(...)` מחזירה את האובייקט האחד שמתאים, ועוצרת בשגיאה כשכמה מתאימים.
- `first()` מחזירה את האובייקט הראשון בתוצאת שאילתה, או `None` כשהיא ריקה.
- מחברת מאפשרת להריץ שאילתה מחוץ ל־`with db_session:` ותוכנית לא; משאירים את הבלוק בשני המקרים.

</div>

<div dir="rtl">

# חלק 2: קשרי Pony ORM: טבלה שמצביעה על טבלה אחרת

דף השיעור של החלק הזה: [קשרי Pony ORM: טבלה שמצביעה על טבלה אחרת](https://dvirross.com/he/courses/intro-data-science/python-orm-pony-relations)

</div>

<div dir="rtl">

## טבלה שמצביעה על טבלה אחרת

שיעור זה נשען על [טבלאות כאובייקטים ב־Python: היכרות ראשונה עם Pony ORM](https://dvirross.com/he/courses/intro-data-science/python-orm-pony), שבו הוצגו ישויות, `with db_session:`, `select`, `get`, פונקציות הסיכום ו־`first()`. כאן אותם כלים עובדים על מסד נתונים חדש, עם נתונים מהבנק העולמי, והישויות שלו מצביעות זו על זו.

ב[מדריך ה־SQL](https://dvirross.com/he/courses/intro-data-science/sql-change-and-join) עמודת המחלקה בטבלה של עובדים מחזיקה את המפתח הראשי של שורה בטבלה אחרת, טבלת המחלקות. עמודה כזו היא [**מפתח זר**](https://dvirross.com/he/courses/intro-data-science/glossary#foreign-key), ו־`JOIN` עוקב אחריה כדי להציב כל שורה ליד השורה שהיא מצביעה עליה. כששורה אחת בטבלה הראשונה מקבלת הצבעה משורות רבות בטבלה השנייה, כמו מדינה אחת שמצביעים עליה המדידות השנתיות שלה, זה [**קשר של אחד לרבים**](https://dvirross.com/he/courses/intro-data-science/glossary#one-to-many-relation). ה־ORM כותב את אותו קישור כתכונות: אובייקט של מדידה מחזיק את אובייקט המדינה שלו, ואובייקט של מדינה מחזיק את כל אובייקטי המדידות שלה.

הנתונים אמיתיים והם מגיעים מ־World Development Indicators של הבנק העולמי (רישיון CC BY 4.0), ונשלפו ב־30 באוקטובר 2026 דרך ה־API של הבנק העולמי. משתמשים בשני מדדים. `SP.DYN.LE00.IN` הוא **תוחלת החיים בלידה**: מספר השנים שיילוד היה חי אילו שיעורי התמותה של אותה שנה נשארו כך כל חייו. `NY.GDP.PCAP.PP.CD` הוא **התוצר לנפש במונחי כוח קנייה** (GDP per capita, PPP): ערך מה שמדינה מייצרת בשנה, לכל נפש, בדולרים בינלאומיים שוטפים, יחידה שמתקנת הבדלי מחירים בין מדינות. זה מבחר קטן: 10 מדינות שנבחרו כך שיכסו אזורים ורמות הכנסה שונים, לשנים 2000 ו־2020. רמת ההכנסה של כל מדינה היא הסיווג של הבנק העולמי עצמו, כפי שפורסם בעת השליפה.

שלוש אזהרות מלוות את הנתונים. מעט מדינות אינן העולם, ולכן המבחר יכול להראות דפוס אבל לא למדוד אותו. שתי מדידות שנעות יחד הן קשר סטטיסטי ולא הוכחה שאחת גורמת לשנייה. בנוסף, ערך שאין למקור נשאר חסר: הוא אינו אפס. התוצר של סוריה ב־2000 חסר במקור, ולכן הוא חסר גם כאן.

השורה הראשונה בתיבה הבאה מביאה את השמות של Pony, ו־`geo` הוא אובייקט מסד הנתונים של השיעור הזה; `':memory:'` הוא מסד נתונים זמני בזיכרון, כמו קודם. כמה שמות מתארים את הקשר ואת סוגי העמודות החדשים:

- `Set('Observation')` היא התכונה `observations` של מדינה. היא מחזיקה את הקבוצה של כל המדידות שמצביעות על המדינה הזו, והיא אינה עמודה בטבלה: Pony קורא אותה מהעמודה שבצד השני. שם המחלקה בגרשיים כי `Observation` מוגדרת מתחתיה.
- `Required(Country)` היא התכונה `country` של מדידה. ערכה חייב להיות אובייקט `Country` אחד, ולכן כל מדידה שייכת למדינה. `Optional(Country)` היתה מאפשרת מדידה בלי מדינה.
- `Optional(float)` היא מספר עם נקודה עשרונית שיכול להיות חסר. ערך חסר הוא `None` ב־Python ו־`NULL` ב־SQL.

הפונקציה `show(Observation)` מדפיסה את התיאור של ישות כפי ש־Pony הבין אותו: התכונות שלה והסוגים שלהן, כולל ה־`id` ש־Pony הוסיף.

</div>

In [31]:
from pony.orm import *

geo = Database()
geo.bind(provider='sqlite', filename=':memory:')

class Country(geo.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(geo.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

geo.generate_mapping(create_tables=True)
show(Observation)

class Observation(Entity):
    id = PrimaryKey(int, auto=True)
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)


<div dir="rtl">

### יוצרים את האובייקטים ומחברים ביניהם

המבחר הוא שתי רשימות של טאפלים: המדינות, והמדידות, וכל מדידה עם קוד המדינה שלה, השנה, תוחלת החיים והתוצר לנפש, כפי שהמקור מפרסם אותם.

</div>

In [32]:
country_rows = [
    ('ISR', 'Israel', 'High income'),
    ('JPN', 'Japan', 'High income'),
    ('USA', 'United States', 'High income'),
    ('NOR', 'Norway', 'High income'),
    ('BRA', 'Brazil', 'Upper middle income'),
    ('CHN', 'China', 'Upper middle income'),
    ('EGY', 'Egypt, Arab Rep.', 'Lower middle income'),
    ('IND', 'India', 'Lower middle income'),
    ('ETH', 'Ethiopia', 'Low income'),
    ('SYR', 'Syrian Arab Republic', 'Low income'),
]
print(len(country_rows))

10


In [33]:
observation_rows = [
    ('ISR', 2000, 78.9536585365854, 25746.8295829392),
    ('ISR', 2020, 82.6487804878049, 40969.2987342816),
    ('JPN', 2000, 81.0760975609756, 27694.5384332959),
    ('JPN', 2020, 84.56, 43560.021428549),
    ('USA', 2000, 76.6365853658537, 36329.9702595751),
    ('USA', 2020, 76.9804878048781, 64465.2971415748),
    ('NOR', 2000, 78.6341463414634, 36813.8379065645),
    ('NOR', 2020, 83.209756097561, 69779.2862630574),
    ('BRA', 2000, 69.584, 9091.76851342904),
    ('BRA', 2020, 74.506, 16101.6183063466),
    ('CHN', 2000, 72.288, 2963.43487212206),
    ('CHN', 2020, 78.019, 18267.4783455827),
    ('EGY', 2000, 67.332, 5451.51566974242),
    ('EGY', 2020, 69.79, 15231.865417764),
    ('IND', 2000, 62.749, 2030.76576304986),
    ('IND', 2020, 70.156, 6766.06119055994),
    ('ETH', 2000, 50.88, 473.903313465592),
    ('ETH', 2020, 65.969, 2406.78938480121),
    ('SYR', 2000, 70.856, None),
    ('SYR', 2020, 71.944, 3878.42802993832),
]
print(len(observation_rows))

20


<div dir="rtl">

ערכים במילון יכולים להיות אובייקטים כלשהם, לא רק מספרים וטקסט. נבנה מילון שהמפתחות שלו הם קודי המדינות והערכים שלו הם אובייקטי `Country`. כדי ליצור מדידה של מדינה, נחפש את אובייקט המדינה לפי הקוד שלה ונעביר אותו כ־`country=...`. כך מחברים בין שתי השורות: Pony שומר את המפתח של המדינה בעמודה `country` של המדידה. הערך `None` בשורה נכנס לעמודה `Optional` כערך חסר.

</div>

In [34]:
with db_session:
    countries = {}
    for code, name, income in country_rows:
        countries[code] = Country(
            code=code, name=name, income_level=income
        )
    for code, year, life, gdp in observation_rows:
        Observation(
            year=year, life_expectancy=life,
            gdp_per_capita=gdp, country=countries[code]
        )
    print(
        Country.select().count(),
        Observation.select().count()
    )

10 20


<div dir="rtl">

### עוקבים אחרי הקשר בשני הכיוונים

בתוך `db_session` נקודה עוקבת אחרי הקשר. עבור מדידה `obs`, הביטוי `obs.country` הוא אובייקט ה־`Country` שלה, ו־`obs.country.name` הוא שם המדינה. בכיוון ההפוך, `country.observations` היא קבוצת המדידות של מדינה. אפשר לעבור על קבוצה בלולאה כמו על רשימה, ויש לה שלוש שיטות: `count()` מחזירה את מספר האובייקטים בה, `order_by(...)` ממיינת אותה כמו בשאילתה, ו־`select(...)` משאירה את האובייקטים שהפונקציה שנותנים לה מחזירה עבורם אמת. כדי לקחת אובייקט אחד לפי מקומו, כמו מרשימה, הופכים קודם את התוצאה של `order_by` או של `select` לרשימה עם `list(...)`.

</div>

In [35]:
with db_session:
    israel = Country.get(code='ISR')
    print(israel.name, israel.income_level)
    print(israel.observations.count())
    in_2020 = list(
        israel.observations.select(lambda o: o.year == 2020)
    )
    obs = in_2020[0]
    print(
        obs.country.name, obs.year,
        round(obs.life_expectancy, 1)
    )

Israel High income
2


Israel 2020 82.6


<div dir="rtl">

הפונקציה `round(x, 1)` מעגלת את `x` לספרה אחת אחרי הנקודה העשרונית. השורות הראשונות הולכות מהמדינה אל המדידות שלה, והשורה האחרונה חוזרת ממדידה אל המדינה שלה.

השיטה `first()` של שאילתה עובדת גם על קבוצה של אובייקטים קשורים. התיבה לוקחת את המדידה האחרונה של ישראל, ואחר כך מחפשת שנה שאין לה מדידה:

</div>

In [36]:
with db_session:
    israel = Country.get(code='ISR')
    latest = israel.observations.order_by(
        lambda o: desc(o.year)
    ).first()
    print(latest.year, round(latest.life_expectancy, 1))
    missing = israel.observations.select(
        lambda o: o.year == 1990
    ).first()
    print(missing)

2020 82.6
None


<div dir="rtl">

הקריאה הראשונה מחזירה מדידה, והשנייה אינה מוצאת מדידה של 1990 ומחזירה `None`.

### שאלות שהאובייקטים הקשורים עונים עליהן

לכל מדינה, `country.observations.order_by(lambda o: o.year)` מחזיקה את שתי המדידות שלה, השנה המוקדמת קודם, ו־`list(...)` הופכת אותה לרשימה, כך שהמקום `[0]` הוא 2000 והמקום `[1]` הוא 2020. השינוי בתוחלת החיים הוא הערך של 2020 פחות הערך של 2000. נאסוף כל שינוי עם שם המדינה שלו ברשימה של טאפלים, ו־`sorted(..., reverse=True)` מציבה את השינוי הגדול ראשון:

</div>

In [37]:
with db_session:
    gains = []
    for country in Country.select():
        years = list(
            country.observations.order_by(lambda o: o.year)
        )
        early = years[0].life_expectancy
        late = years[1].life_expectancy
        gains.append((round(late - early, 1), country.name))
    ranked = sorted(gains, reverse=True)
    print(ranked[:3])
    print(ranked[-1])

[(15.1, 'Ethiopia'), (7.4, 'India'), (5.7, 'China')]
(0.3, 'United States')


<div dir="rtl">

תוחלת החיים עלתה בכל מדינה במבחר, והכי הרבה באתיופיה, שהתחילה מהנמוכה ביותר. המבחר אינו אומר מדוע. שתי השנים הן תצלום מצב, והשינוי הקטן ביותר, 0.3 שנים בארצות הברית, יכול לנבוע מהמגמה הארוכה, מכך ש־2020 היא שנת המגפה, או משניהם; שתי נקודות אינן יכולות להפריד ביניהם.

בתוצר לנפש ערך יכול להיות חסר, ואז אין מה לחלק. הבדיקה `x is None` מתקיימת כש־`x` הוא הערך החסר, ולכן הלולאה מפרידה את המדינות שאין להן ערך, ולשאר היא מחשבת פי כמה התוצר גדל: הערך של 2020 חלקי הערך של 2000.

</div>

In [38]:
with db_session:
    multiples = []
    missing = []
    for country in Country.select():
        years = list(
            country.observations.order_by(lambda o: o.year)
        )
        early = years[0].gdp_per_capita
        late = years[1].gdp_per_capita
        if early is None or late is None:
            missing.append(country.name)
        else:
            multiples.append(
                (round(late / early, 1), country.name)
            )
    print(sorted(multiples, reverse=True)[:3])
    print('missing:', missing)

[(6.2, 'China'), (5.1, 'Ethiopia'), (3.3, 'India')]
missing: ['Syrian Arab Republic']


<div dir="rtl">

סין ואתיופיה הכפילו את התוצר לנפש שלהן ביותר מפי חמישה, מרמות נמוכות, ולכן הכפולות שלהן גדולות, בעוד שהתוספת לנפש באתיופיה (כ־1,900 דולר) קטנה ליד התוספת של נורווגיה (כ־33,000), שאינה ברשימה. כפולה והפרש עונים על שתי שאלות שונות. סוריה אינה בדירוג כי הערך שלה ל־2000 חסר, ולא כי הוא אפס.

### תנאי שחוצה את הקשר

בתוך שאילתה נקודה יכולה לעקוב אחרי הקשר גם כן: `o.country.income_level` היא רמת ההכנסה של המדינה של המדידה `o`. שאילתה כזו צריכה את שתי הטבלאות, ו־Pony כותב את החיבור שמדריך ה־SQL כתב ביד. נקבץ את המדידות של 2020 לפי רמת הכנסה ונחשב ממוצע של שני המדדים; `avg` מדלגת על ערכים חסרים:

</div>

In [39]:
with db_session:
    by_level = select(
        (
            o.country.income_level,
            avg(o.life_expectancy),
            avg(o.gdp_per_capita)
        )
        for o in Observation if o.year == 2020
    )
    print(by_level.get_sql())
    by_gdp = sorted(by_level, key=lambda row: row[2])
    for level, life, gdp in by_gdp:
        print(level, round(life, 1), round(gdp))

SELECT "country"."income_level", AVG("o"."life_expectancy"), AVG("o"."gdp_per_capita")
FROM "Observation" "o", "Country" "country"
WHERE "o"."year" = 2020
  AND "o"."country" = "country"."id"
GROUP BY "country"."income_level"
Low income 69.0 3143
Lower middle income 70.0 10999
Upper middle income 76.3 17185
High income 81.8 54693


<div dir="rtl">

ב־`WHERE` יש את התנאי על השנה ואת השורה שמחברת את המפתח הזר `"o"."country"` למפתח הראשי `"country"."id"`, כמו ש־`ON` עושה ב־`JOIN`. במבחר הזה רמות הכנסה גבוהות יותר מלוות תוחלת חיים ארוכה יותר, וזה קשר סטטיסטי בין ממוצעים של שתיים עד ארבע מדינות בכל קבוצה. הוא אינו מראה שהכנסה מייצרת בריאות, או להפך.

### ה־SQL שרץ בפועל

השיטה `get_sql` מראה את הטקסט של שאילתה אחת. הפונקציה `set_sql_debug(True)` רחבה יותר: מאותו רגע Pony מדפיס כל פקודת SQL ברגע ששולח אותה למסד הנתונים, עם כמה שורות על החיבור, עד ש־`set_sql_debug(False)` עוצרת אותו. כאן היא מראה מה עולה מעקב אחרי קשר:

</div>

In [40]:
with db_session:
    set_sql_debug(True)
    norway = Country.get(code='NOR')
    print(norway.observations.count())
    set_sql_debug(False)

GET CONNECTION FROM THE LOCAL POOL
SWITCH TO AUTOCOMMIT MODE
SELECT "id", "code", "name", "income_level"
FROM "Country"
WHERE "code" = ?
LIMIT 2
['NOR']

SELECT COUNT(*)
FROM "Observation"
WHERE "country" = ?
[4]

2


<div dir="rtl">

### מסד נתונים שכבר קיים

עד כאן `':memory:'` נתן מסד חדש וריק בכל פעם. קובץ מסד נתונים שכבר קיים, שנוצר בתוכנית אחרת, מתחברים אליו לפי שם הקובץ: `geo.bind(provider='sqlite', filename='wdi_extract.db')`. בלי הארגומנט `create_db=True`, Pony אינו יוצר את הקובץ, ולכן שם קובץ שגוי הוא שגיאה ולא מסד חדש וריק. הקריאה `generate_mapping()` בלי `create_tables=True` אינה יוצרת דבר: היא בודקת שהטבלאות של הקובץ קיימות ושיש בהן את העמודות שהמחלקות מתארות, ועוצרת עם שגיאה אם הן אינן תואמות. לכן המחלקות צריכות לתאר את הטבלאות כפי שהן, עם אותם שמות.

הקטע שלהלן מוצג ואינו רץ, כי הוא צריך את הקובץ. יצרנו את הקובץ `wdi_extract.db` מראש, מהמחלקות של הפרק הזה עם `create_db=True`:

```python
geo = Database()
geo.bind(provider='sqlite', filename='wdi_extract.db')

class Country(geo.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(geo.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

geo.generate_mapping()

with db_session:
    israel = Country.get(code='ISR')
    print(israel.name, israel.observations.count())
```

```text
Israel 2
```

### שורות מקובץ

טבלה מגיעה לא פעם כ[קובץ CSV](https://dvirross.com/he/courses/intro-data-science/python-values-and-collections#סוגי-קבצי-נתונים): קובץ טקסט עם שורה אחת בכל שורה ועם שמות העמודות בשורה הראשונה. הקובץ `world-bank-more-countries.csv` מכיל את אותם שני מדדים לארבע מדינות נוספות, לשנים 2000 ו־2020, עם שורה לכל מדינה ושנה. המקור והרישיון הם של התמצית שלמעלה; הקובץ נשלף ב־5 בנובמבר 2026. התוצר לנפש של אריתריאה ב־2020 חסר במקור, ולכן השדה הזה בקובץ ריק.

Python קוראת קובץ כזה עם המודול `csv`. הפונקציה `open(name, newline='', encoding='utf-8')` פותחת קובץ, ובלוק `with` סוגר אותו כשהבלוק מסתיים; האפשרות `newline=''` היא מה שהמודול `csv` מבקש. הפונקציה `csv.DictReader(file)` עוברת על השורות שאחרי הראשונה, ונותנת כל שורה כ[מילון](https://dvirross.com/he/courses/intro-data-science/glossary#dictionary) שהמפתחות שלו הם שמות העמודות. כל ערך שמגיע מקובץ הוא טקסט, גם מספר, ושדה ריק הוא הטקסט הריק `''`. הפונקציה `int(text)` הופכת טקסט למספר שלם, ו־`float(text)` הופכת טקסט למספר עם נקודה עשרונית.

</div>

In [41]:
import csv

with open(
    'world-bank-more-countries.csv', newline='', encoding='utf-8'
) as file:
    rows = list(csv.DictReader(file))

print(len(rows))
print(rows[0]['name'], rows[0]['year'] * 2)
print(rows[0]['name'], int(rows[0]['year']) + 1)
print(rows[7]['gdp_per_capita'] == '')

8
Chile 20002000
Chile 2001
True


<div dir="rtl">

השנה היא טקסט, ולכן חזרה עליה פעמיים מדביקה שני עותקים של הטקסט, ורק `int(...)` הופכת אותה למספר שאפשר לחבר אליו. ערך חסר אסור שייהפך למספר: שדה ריק הופך ל־`None` בביטוי תנאי, `float(text) if text != '' else None`, שהוא `None` כשהשדה ריק ומספר בכל מקרה אחר.

טעינת השורות לישויות היא הלולאה של [יוצרים את האובייקטים ומחברים ביניהם](#יוצרים-את-האובייקטים-ומחברים-ביניהם), עם `row['code']` במקום הקוד המחולץ. `Country` נוצרת רק בפעם הראשונה שהקוד שלה מופיע: הבדיקה `code not in countries` מתקיימת כשבמילון אין עדיין מפתח `code`. השיטה `get` מקבלת כמה תנאים בבת אחת: `Observation.get(year=2020, country=some_country)` מחזירה את המדידה האחת שמתאימה לכולם.

### תרגול

**נסו בעצמכם.** צרו אובייקט מסד נתונים `wb_db` בזיכרון, כמו בפרק הזה. אחר כך כתבו שתי ישויות, בסדר הזה. ל־`Economy` יש את התכונות `code` ו־`name`, כל אחת טקסט שחייב להיות לו ערך, ואת `readings`, קבוצה של אובייקטי `Reading`. ל־`Reading` יש `indicator` (טקסט שחייב להיות לו ערך), `year` (מספר שלם שחייב להיות לו ערך), `value` (מספר עם נקודה עשרונית שיכול להיות חסר) ו־`economy` (אובייקט `Economy` שחייב להיות לו ערך). אחר כך צרו את הטבלאות עם `generate_mapping` והדפיסו `show(Reading)`.

</div>

<div dir="rtl">

> **רמז**
>
> כל מחלקה יורשת מ־`wb_db.Entity`. את `Reading` כותבים בתוך `Set(...)` בגרשיים, כי היא מוגדרת מתחת ל־`Economy`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [42]:
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)
show(Reading)

class Reading(Entity):
    id = PrimaryKey(int, auto=True)
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)


<div dir="rtl">

**פלט צפוי**

```text
class Reading(Entity):
    id = PrimaryKey(int, auto=True)
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)
```

</div>

In [43]:
check(8)

OK: הפלט כצפוי.


<div dir="rtl">

התכונה `value` היא `Optional(float)`, כי למקור יכול לחסר ערך, והתכונה `economy` היא `Required(Economy)`, ולכן כל קריאה מצביעה על כלכלה.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Required(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)
show(Reading)
```

התכונה `value` היא `Required(float)`, ולכן אי אפשר לשמור קריאה עם ערך חסר. המשימה מבקשת `Optional`.

</div>

<div dir="rtl">

**נסו בעצמכם.** המחלקות `Economy` ו־`Reading` שלהלן מוכנות, `names` מחזיק את השם של כל קוד, ו־`rows` מחזיק קריאות אמיתיות של שתי כלכלות: דרום אפריקה (`ZAF`) וסוריה (`SYR`), שהתוצר שלה ל־2000 חסר במקור. בתוך בלוק `with db_session:` צרו מילון `economies` שהמפתחות שלו הם הקודים והערכים שלו אובייקטי `Economy`, ואז צרו `Reading` לכל שורה, מחובר לכלכלה שלו דרך המילון. אחר כך הדפיסו, לכל קוד במילון, את הקוד ואת מספר הקריאות שלו, עם `count()`.

</div>

<div dir="rtl">

> **רמז**
>
> בתוך בלוק `with db_session:`, עברו קודם בלולאה על `names` ושימו במילון `Economy(code=..., name=...)` תחת כל קוד. אחר כך עברו בלולאה על `rows` והעבירו `economy=economies[code]` ל־`Reading`. לולאה אחרונה על `economies` מדפיסה את הקוד ואת `economies[code].readings.count()`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [44]:
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)
names = {'ZAF': 'South Africa', 'SYR': 'Syrian Arab Republic'}
rows = [
    ('ZAF', 'life', 2000, 58.368),
    ('ZAF', 'life', 2020, 65.15),
    ('ZAF', 'gdp', 2020, 12671.0862436851),
    ('SYR', 'life', 2000, 70.856),
    ('SYR', 'life', 2020, 71.944),
    ('SYR', 'gdp', 2000, None),
    ('SYR', 'gdp', 2020, 3878.42802993832),
]

with db_session:
    economies = {}
    for code in names:
        economies[code] = Economy(code=code, name=names[code])
    for code, indicator, year, value in rows:
        Reading(
            indicator=indicator, year=year, value=value,
            economy=economies[code]
        )
    for code in economies:
        print(code, economies[code].readings.count())

ZAF 3
SYR 4


<div dir="rtl">

**פלט צפוי**

```text
ZAF 3
SYR 4
```

</div>

In [45]:
check(9)

OK: הפלט כצפוי.


<div dir="rtl">

לדרום אפריקה יש שלוש קריאות ולסוריה ארבע, אחת מהן עם ערך חסר, שנשמר כשורה ולא מדולג.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    economies = {}
    for code in names:
        economies[code] = Economy(code=code, name=names[code])
    for code, indicator, year, value in rows:
        Reading(
            indicator=indicator, year=year, value=value,
            economy=economies['ZAF']
        )
    for code in economies:
        print(code, economies[code].readings.count())
```

במילון מבקשים תמיד את `'ZAF'`, ולכן כל הקריאות מחוברות לדרום אפריקה ולסוריה אין אף אחת. המפתח צריך לבוא מהשורה: `economies[code]`.

</div>

<div dir="rtl">

**נסו בעצמכם.** התיבה שלהלן שומרת את הכלכלה עם הקוד `'ZAF'` ושלוש מהקריאות שלה. בבלוק `with db_session:` חדש, קבלו את הכלכלה הזו עם `Economy.get(code='ZAF')` והדפיסו את השינוי בתוחלת החיים שלה בין 2000 ל־2020, מעוגל לספרה אחת אחרי הנקודה העשרונית: הערך של 2020 פחות הערך של 2000, לפי הקריאות עם המדד `'life'`.

</div>

<div dir="rtl">

> **רמז**
>
> לקבוצה `readings` של הכלכלה יש את השיטות `select` ו־`order_by`. השאירו את הקריאות שהמדד שלהן הוא `'life'`, מיינו אותן לפי שנה, והפחיתו את הערך במקום `[0]` מהערך במקום `[1]`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [46]:
wb_db = Database()
wb_db.bind(provider='sqlite', filename=':memory:')

class Economy(wb_db.Entity):
    code = Required(str)
    name = Required(str)
    readings = Set('Reading')

class Reading(wb_db.Entity):
    indicator = Required(str)
    year = Required(int)
    value = Optional(float)
    economy = Required(Economy)

wb_db.generate_mapping(create_tables=True)

rows = [
    ('life', 2020, 65.15),
    ('life', 2000, 58.368),
    ('gdp', 2020, 12671.0862436851),
]

with db_session:
    zaf = Economy(code='ZAF', name='South Africa')
    for indicator, year, value in rows:
        Reading(
            indicator=indicator, year=year,
            value=value, economy=zaf
        )

with db_session:
    zaf = Economy.get(code='ZAF')
    life = zaf.readings.select(
        lambda r: r.indicator == 'life'
    )
    years = list(life.order_by(lambda r: r.year))
    print(round(years[1].value - years[0].value, 1))

6.8


<div dir="rtl">

**פלט צפוי**

```text
6.8
```

</div>

In [47]:
check(10)

OK: הפלט כצפוי.


<div dir="rtl">

שתי הקריאות נוצרו כשהשנה המאוחרת קודמת, ולכן המיון לפי שנה הוא שמציב את 2000 במקום `[0]`.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
with db_session:
    zaf = Economy.get(code='ZAF')
    life = list(
        zaf.readings.select(lambda r: r.indicator == 'life')
    )
    print(round(life[1].value - life[0].value, 1))
```

בלי `order_by`, הקריאות מגיעות בסדר שבו נוצרו, וכאן השנה המאוחרת נוצרה ראשונה, ולכן החיסור מתבצע בכיוון ההפוך והסימן מתהפך.

</div>

<div dir="rtl">

**נסו בעצמכם.** המחלקות `Country` ו־`Observation` על מסד הנתונים `file_db` שלהלן מוכנות, והקובץ `world-bank-more-countries.csv` נמצא בתיקיית העבודה. קראו את הקובץ עם `csv.DictReader`. בבלוק `with db_session:` צרו מילון `countries` מכל קוד לאובייקט ה־`Country` שלו, כשכל מדינה נוצרת פעם אחת, וצרו `Observation` לכל שורה, עם השנה כמספר שלם ושני הערכים כמספרים, או `None` כששדה ריק. הדפיסו את מספר המדינות ואת מספר המדידות בשורה אחת, עם רווח ביניהם, ובשורה הבאה את התוצר לנפש של אריתריאה (`ERI`) ב־2020.

</div>

<div dir="rtl">

> **רמז**
>
> ייבאו את `csv`, פתחו את הקובץ כמו בתיבה שלמעלה, ובנו רשימה של השורות עם `list(csv.DictReader(file))`. בבלוק עברו על השורות בלולאה: צרו את ה־`Country` רק כשהקוד שלה עדיין אינו מפתח ב־`countries`, ואת ה־`Observation` צרו תמיד, עם `country=countries[code]`. השורה האחרונה היא `Observation.get(...)` עם המדינה והשנה, ואז התכונה `gdp_per_capita`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [48]:
import csv

file_db = Database()
file_db.bind(provider='sqlite', filename=':memory:')

class Country(file_db.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(file_db.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

file_db.generate_mapping(create_tables=True)

with open(
    'world-bank-more-countries.csv', newline='', encoding='utf-8'
) as file:
    rows = list(csv.DictReader(file))

with db_session:
    countries = {}
    for row in rows:
        code = row['code']
        if code not in countries:
            countries[code] = Country(
                code=code, name=row['name'],
                income_level=row['income_level']
            )
        life = row['life_expectancy']
        gdp = row['gdp_per_capita']
        Observation(
            year=int(row['year']),
            life_expectancy=float(life) if life != '' else None,
            gdp_per_capita=float(gdp) if gdp != '' else None,
            country=countries[code]
        )
    print(Country.select().count(), Observation.select().count())
    eritrea = Observation.get(
        country=countries['ERI'], year=2020
    )
    print(eritrea.gdp_per_capita)

4 8
None


<div dir="rtl">

**פלט צפוי**

```text
4 8
None
```

</div>

In [49]:
check(11)

OK: הפלט כצפוי.


<div dir="rtl">

נשמרו ארבע מדינות ושמונה מדידות, והשדה הריק של אריתריאה הפך ל־`None` ולא למספר.

</div>

<div dir="rtl">

**טעות נפוצה**

```python
import csv

file_db = Database()
file_db.bind(provider='sqlite', filename=':memory:')

class Country(file_db.Entity):
    code = Required(str)
    name = Required(str)
    income_level = Required(str)
    observations = Set('Observation')

class Observation(file_db.Entity):
    year = Required(int)
    life_expectancy = Optional(float)
    gdp_per_capita = Optional(float)
    country = Required(Country)

file_db.generate_mapping(create_tables=True)

with open(
    'world-bank-more-countries.csv', newline='', encoding='utf-8'
) as file:
    rows = list(csv.DictReader(file))

with db_session:
    countries = {}
    for row in rows:
        code = row['code']
        countries[code] = Country(
            code=code, name=row['name'],
            income_level=row['income_level']
        )
        life = row['life_expectancy']
        gdp = row['gdp_per_capita']
        Observation(
            year=int(row['year']),
            life_expectancy=float(life) if life != '' else None,
            gdp_per_capita=float(gdp) if gdp != '' else None,
            country=countries[code]
        )
    print(Country.select().count(), Observation.select().count())
    eritrea = Observation.get(
        country=countries['ERI'], year=2020
    )
    print(eritrea.gdp_per_capita)
```

`Country` נוצרת לכל שורה, בלי הבדיקה `code not in countries`, ולכן יש שמונה מדינות ולא ארבע, שתיים לכל קוד. המילון שומר רק את האחרונה מכל קוד, ולכן השורה השנייה עדיין נראית נכונה.

</div>

<div dir="rtl">

## לזכור

- **קשר של אחד לרבים** הוא מפתח זר שנראה משני הצדדים: `Required(Country)` או `Optional(Country)` בצד של הרבים, ו־`Set('Observation')` בצד של האחד. נקודה עוקבת אחריו בשני הכיוונים, ובתוך שאילתה נקודה כמו `o.country.income_level` גורמת ל־Pony לכתוב את החיבור.
- `Optional(float)` מאפשרת ערך חסר, שהוא `None`; בודקים אותו עם `is None` לפני שמחשבים. ערך חסר אינו אפס.
- מילון מקוד לאובייקט (`countries[code]`) מחבר אובייקטים חדשים לאובייקט הקשור הנכון.
- `csv.DictReader` קוראת קובץ CSV: כל ערך הוא טקסט, ולכן ממירים אותו עם `int(...)` או `float(...)`, ושדה ריק הופך ל־`None`.
- `show(Entity)` מדפיסה איך Pony הבין ישות, ו־`set_sql_debug(True)` מדפיסה כל פקודת SQL שרצה. כדי להשתמש בקובץ מסד נתונים שקיים, מחברים אותו עם `bind` בלי `create_db=True` וקוראים ל־`generate_mapping()` בלי `create_tables=True`.

</div>